# 基于 GRPO 的 Qwen3.5 小模型理科推理能力增强研究
## Kaggle Notebook — 双 T4 一键运行

> **Run All** 从头跑完：SFT 预热 → GRPO (GSM8K) → GRPO (SciBench 7科) → 消融 → 评估

## ⚙️ 参数配置

In [ ]:
# ============================================================
# 模型
# ============================================================
MODEL_HF = "Qwen/Qwen3.5-0.8B"
# MODEL_HF = "Qwen/Qwen3.5-1.7B"  # 更大模型，效果更好但更慢

# ============================================================
# 训练数据
# ============================================================
GSM8K_DATASET = "openai/gsm8k"
GSM8K_CONFIG  = "main"
SCIBENCH_DATASET = "xw27/scibench"

# SciBench 科目分割: 7 科训练 / 3 科测试 (held-out, 零泄漏)
SCIBENCH_TRAIN_SUBJECTS = ["atkins","calculus","chemmc","class","diff","fund","matter"]
SCIBENCH_TEST_SUBJECTS  = ["quan","stat","thermo"]

# ============================================================
# 评估开关
# ============================================================
EVAL_GSM8K    = True
EVAL_SCIBENCH = True

# ============================================================
# SFT 参数
# ============================================================
SFT_EPOCHS      = 2
SFT_BATCH       = 4
SFT_GRAD_ACCUM  = 4
SFT_LR          = 2e-4
SFT_LORA_R      = 16
SFT_LORA_ALPHA  = 32
SFT_MAX_SEQ     = 1024
SFT_MAX_SAMPLES = 4000

# ============================================================
# GRPO 参数
# ============================================================
GRPO_EPOCHS        = 2
GRPO_BATCH         = 2
GRPO_GRAD_ACCUM    = 8
GRPO_LR            = 5e-5
GRPO_LORA_R        = 16
GRPO_LORA_ALPHA    = 32
GRPO_NUM_GENERATIONS = 8
GRPO_KL_COEF       = 0.04
GRPO_TEMPERATURE   = 1.0
GRPO_MAX_PROMPT     = 512
GRPO_MAX_COMPLETION = 512
GRPO_MAX_SAMPLES   = 2000

# SciBench GRPO (在 GSM8K GRPO 基础上追加)
SCIBENCH_GRPO_EPOCHS     = 1
SCIBENCH_GRPO_MAX_SAMPLES = 500

# ============================================================
# 评估参数
# ============================================================
EVAL_MAX_SAMPLES    = 200
EVAL_MAX_NEW_TOKENS = 1024

# ============================================================
# 开关 + 路径
# ============================================================
RUN_ABLATIONS = True
OUTPUT_ROOT   = "/kaggle/working/outputs"

print("✅ 参数就绪")
print(f"   模型: {MODEL_HF}")
print(f"   SciBench 训练: {len(SCIBENCH_TRAIN_SUBJECTS)} 科 | 测试: {len(SCIBENCH_TEST_SUBJECTS)} 科 (held-out)")

## Step 1: 安装依赖

In [ ]:
import subprocess, sys
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs),
                   capture_output=True)

pip("transformers>=4.47.0", "accelerate>=1.0.0")
pip("peft>=0.13.0", "bitsandbytes>=0.44.0", "trl>=0.12.0")
pip("datasets>=3.0.0", "numpy", "pandas", "matplotlib", "seaborn", "tqdm")
pip("wandb", "tensorboard", "gradio>=5.0.0", "modelscope")

print("✅ 依赖安装完成")

## Step 2: 环境初始化 + 克隆项目 + 预缓存数据

In [ ]:
import os, sys, json, time, re
from pathlib import Path
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset, load_dataset

# ---- GPU ----
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}, {p.total_mem/1024**3:.1f} GB")

# ---- 克隆 / 更新项目 ----
REPO = "https://github.com/JKpink/term-end-project.git"
PROJECT_DIR = Path("/kaggle/working/term-end-project")
if not (PROJECT_DIR / "project").exists():
    !git clone -q {REPO} {PROJECT_DIR}
    print("✅ 项目已克隆")
else:
    !cd {PROJECT_DIR} && git stash -q && git pull -q && git stash pop -q
    print("✅ 项目已更新 (git pull)")

SRC = PROJECT_DIR / "project" / "src"
os.chdir(str(PROJECT_DIR / "project"))
sys.path.insert(0, str(SRC))
pip("torch", "transformers", "datasets", "accelerate", "peft", "bitsandbytes", "trl", "wandb", "tensorboard", "gradio", "modelscope")
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
print(f"✅ 项目就绪 | 源码: {SRC} | 输出: {OUTPUT_ROOT}")

# ---- 预缓存: ModelScope → HF → HF mirror ----
def cache_dataset(hf_name, ms_id=None, hf_config=None):
    if ms_id:
        try:
            from modelscope import MsDataset
            MsDataset.load(ms_id)
            print(f"  ✅ {hf_name} (ModelScope)")
            return
        except Exception: pass
    try:
        ds = load_dataset(hf_name, hf_config) if hf_config else load_dataset(hf_name)
        n = sum(len(v) for v in ds.values()) if hasattr(ds,'values') else len(ds)
        print(f"  ✅ {hf_name}: {n} 条 (HF)")
        return
    except Exception: pass
    try:
        os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
        ds = load_dataset(hf_name, hf_config) if hf_config else load_dataset(hf_name)
        n = sum(len(v) for v in ds.values()) if hasattr(ds,'values') else len(ds)
        print(f"  ✅ {hf_name}: {n} 条 (HF mirror)")
    except Exception as e:
        print(f"  ❌ {hf_name}: {str(e)[:80]}")

print("\n预缓存...")
cache_dataset("openai/gsm8k", ms_id="swift/gsm8k", hf_config="main")
cache_dataset("xw27/scibench")
print("✅ 环境就绪")

---
## Phase 1: SFT 预热 — GSM8K

QLoRA 监督微调，学习 `<think>推理步骤</think>答案：...` 格式。

In [ ]:
SFT_OUTPUT = f"{OUTPUT_ROOT}/sft_kaggle"

sft_cmd = (
    f"python {SRC}/train_sft.py"
    f" --model_name {MODEL_HF}"
    f" --dataset_name {GSM8K_DATASET}"
    f" --dataset_config {GSM8K_CONFIG}"
    f" --output_dir {SFT_OUTPUT}"
    f" --num_epochs {SFT_EPOCHS}"
    f" --per_device_batch_size {SFT_BATCH}"
    f" --gradient_accumulation_steps {SFT_GRAD_ACCUM}"
    f" --learning_rate {SFT_LR}"
    f" --lora_r {SFT_LORA_R}"
    f" --lora_alpha {SFT_LORA_ALPHA}"
    f" --max_seq_length {SFT_MAX_SEQ}"
    f" --logging_steps 10 --save_steps 200 --use_flash_attention"
)
if SFT_MAX_SAMPLES:
    sft_cmd += f" --max_train_samples {SFT_MAX_SAMPLES}"

print(f"GSM8K | Epochs={SFT_EPOCHS} | Batch={SFT_BATCH*SFT_GRAD_ACCUM} | LoRA r={SFT_LORA_R}")
t0 = time.time()
!{sft_cmd}
print(f"耗时: {(time.time()-t0)/60:.1f} min")

SFT_ADAPTER = f"{SFT_OUTPUT}/final"
assert os.path.exists(SFT_ADAPTER), f"SFT 失败: {SFT_ADAPTER}"
print(f"✅ SFT → {SFT_ADAPTER}")

---
## Phase 2: GRPO — GSM8K (Baseline ④)

SFT 适配器初始化 + GRPO 强化学习，仅用 GSM8K。

In [ ]:
GRPO_GSM8K_OUTPUT = f"{OUTPUT_ROOT}/grpo_gsm8k"

grpo_cmd = (
    f"python {SRC}/train_grpo.py"
    f" --model_name {MODEL_HF}"
    f" --sft_adapter_path {SFT_ADAPTER}"
    f" --output_dir {GRPO_GSM8K_OUTPUT}"
    f" --dataset_name {GSM8K_DATASET}"
    f" --dataset_config {GSM8K_CONFIG}"
    f" --reward_type full"
    f" --num_generations {GRPO_NUM_GENERATIONS}"
    f" --kl_coef {GRPO_KL_COEF}"
    f" --temperature {GRPO_TEMPERATURE}"
    f" --learning_rate {GRPO_LR}"
    f" --lora_r {GRPO_LORA_R}"
    f" --lora_alpha {GRPO_LORA_ALPHA}"
    f" --num_train_epochs {GRPO_EPOCHS}"
    f" --per_device_batch_size {GRPO_BATCH}"
    f" --gradient_accumulation_steps {GRPO_GRAD_ACCUM}"
    f" --max_prompt_length {GRPO_MAX_PROMPT}"
    f" --max_completion_length {GRPO_MAX_COMPLETION}"
    f" --logging_steps 5 --save_steps 100"
)
if GRPO_MAX_SAMPLES:
    grpo_cmd += f" --max_train_samples {GRPO_MAX_SAMPLES}"

print(f"GSM8K | G={GRPO_NUM_GENERATIONS} | KL={GRPO_KL_COEF} | Epochs={GRPO_EPOCHS}")
t0 = time.time()
!{grpo_cmd}
print(f"耗时: {(time.time()-t0)/60:.1f} min")

GRPO_GSM8K_ADAPTER = f"{GRPO_GSM8K_OUTPUT}/final_grpo"
assert os.path.exists(GRPO_GSM8K_ADAPTER), f"GRPO GSM8K 失败"
print(f"✅ Baseline ④ → {GRPO_GSM8K_ADAPTER}")

---
## Phase 3: SciBench 分割 + GRPO — SciBench (Baseline ⑤ 核心)

3 科 held-out 只测试。7 科用于 GRPO 追加训练。

In [ ]:
# ============================================================
# 1. 加载 SciBench 并按科目分割
# ============================================================
print("加载 SciBench...")
scibench_raw = load_dataset("xw27/scibench")
scibench_all = scibench_raw["train"] if "train" in scibench_raw else list(scibench_raw.values())[0]
print(f"总样本: {len(scibench_all)}")

# 方案 A: 从 HF 下载原始 JSON 按文件名区分科目
import requests
BASE = "https://huggingface.co/datasets/xw27/scibench/resolve/main"
ALL_SUBJECTS = ["atkins","calculus","chemmc","class","diff","fund","matter","quan","stat","thermo"]

subjects = {}
download_ok = False
for fname in ALL_SUBJECTS:
    try:
        resp = requests.get(f"{BASE}/{fname}.json", timeout=30)
        if resp.status_code == 200:
            data = resp.json()
            if isinstance(data, list):
                for item in data:
                    item["_subject"] = fname
                subjects[fname] = data
                print(f"  {fname}: {len(data)} 题")
                download_ok = True
    except Exception as e:
        print(f"  {fname}: 下载失败 ({str(e)[:40]})")

# 方案 B: 如果 requests 全部失败，用 load_dataset 的结果随机 7:3 分割
if not download_ok:
    print("\n⚠️  HF 原始文件下载失败，使用随机 7:3 分割作为 fallback")
    import random
    random.seed(42)
    data_list = [dict(ex) for ex in scibench_all]
    random.shuffle(data_list)
    split = int(len(data_list) * 0.7)
    subjects["_train"] = data_list[:split]
    subjects["_test"]  = data_list[split:]
    SCIBENCH_TRAIN_SUBJECTS = ["_train"]
    SCIBENCH_TEST_SUBJECTS  = ["_test"]
    print(f"  训练: {len(subjects['_train'])} 题 | 测试: {len(subjects['_test'])} 题")

# ============================================================
# 2. 按配置分割
# ============================================================
train_data, test_data = [], []
for sname, items in subjects.items():
    if sname in SCIBENCH_TRAIN_SUBJECTS:
        train_data.extend(items)
        print(f"  📚 {sname}: {len(items)} → 训练")
    elif sname in SCIBENCH_TEST_SUBJECTS:
        test_data.extend(items)
        print(f"  🧪 {sname}: {len(items)} → 测试 (held-out)")

print(f"\n训练: {len(train_data)} 题 | 测试: {len(test_data)} 题")

SCIBENCH_TRAIN_FILE = f"{OUTPUT_ROOT}/scibench_train.json"
SCIBENCH_TEST_FILE  = f"{OUTPUT_ROOT}/scibench_test.json"
with open(SCIBENCH_TRAIN_FILE, "w") as f:
    json.dump(train_data, f, ensure_ascii=False)
with open(SCIBENCH_TEST_FILE, "w") as f:
    json.dump(test_data, f, ensure_ascii=False)

In [ ]:
# ============================================================
# 3. GRPO on SciBench (从 GSM8K GRPO 初始化)
# ============================================================
GRPO_MIXED_OUTPUT = f"{OUTPUT_ROOT}/grpo_mixed"
GRPO_MIXED_ADAPTER = None

if len(train_data) > 0:
    print(f"\nGRPO on SciBench ({len(train_data)} 题)...")
    scibench_cmd = (
        f"python {SRC}/train_grpo.py"
        f" --model_name {MODEL_HF}"
        f" --sft_adapter_path {GRPO_GSM8K_ADAPTER}"
        f" --output_dir {GRPO_MIXED_OUTPUT}"
        f" --dataset_name {SCIBENCH_DATASET}"
        f" --reward_type full"
        f" --num_generations {GRPO_NUM_GENERATIONS}"
        f" --kl_coef {GRPO_KL_COEF}"
        f" --temperature {GRPO_TEMPERATURE}"
        f" --learning_rate {GRPO_LR}"
        f" --lora_r {GRPO_LORA_R}"
        f" --lora_alpha {GRPO_LORA_ALPHA}"
        f" --num_train_epochs {SCIBENCH_GRPO_EPOCHS}"
        f" --per_device_batch_size {GRPO_BATCH}"
        f" --gradient_accumulation_steps {GRPO_GRAD_ACCUM}"
        f" --max_prompt_length {GRPO_MAX_PROMPT}"
        f" --max_completion_length {GRPO_MAX_COMPLETION}"
        f" --max_train_samples {SCIBENCH_GRPO_MAX_SAMPLES}"
        f" --logging_steps 5 --save_steps 100"
    )
    print(f"SciBench 7科 | G={GRPO_NUM_GENERATIONS} | Epochs={SCIBENCH_GRPO_EPOCHS}")
    t0 = time.time()
    !{scibench_cmd}
    print(f"耗时: {(time.time()-t0)/60:.1f} min")

    GRPO_MIXED_ADAPTER = f"{GRPO_MIXED_OUTPUT}/final_grpo"
    if os.path.exists(GRPO_MIXED_ADAPTER):
        print(f"✅ Baseline ⑤ → {GRPO_MIXED_ADAPTER}")
    else:
        print("⚠️  Baseline ⑤ 未生成")
else:
    print("❌ SciBench 训练数据为空")

print(f"\nBaseline: ④={'✅' if GRPO_GSM8K_ADAPTER else '❌'} ⑤={'✅' if GRPO_MIXED_ADAPTER else '❌'}")

---
## Phase 4: 消融实验

7 组，每组 1 epoch / 500 样本。try/except 保护。

In [ ]:
ablation_results = {}

if not RUN_ABLATIONS:
    print("⏭️  跳过消融")
else:
    ablations = [
        ("abl_no_sft",      "full",                    False, GRPO_NUM_GENERATIONS, GRPO_KL_COEF),
        ("abl_correctness", "correctness_only",         True,  GRPO_NUM_GENERATIONS, GRPO_KL_COEF),
        ("abl_format",      "correctness_and_format",   True,  GRPO_NUM_GENERATIONS, GRPO_KL_COEF),
        ("abl_g4",          "full",                    True,  4,                      GRPO_KL_COEF),
        ("abl_g16",         "full",                    True,  16,                     GRPO_KL_COEF),
        ("abl_kl001",       "full",                    True,  GRPO_NUM_GENERATIONS,   0.01),
        ("abl_kl10",        "full",                    True,  GRPO_NUM_GENERATIONS,   0.10),
    ]

    for name, reward_type, use_sft, gsize, kl in ablations:
        print(f"\n{'='*50} 消融: {name}  {'='*50}")
        try:
            outdir = f"{OUTPUT_ROOT}/grpo_{name}"
            cmd = (
                f"python {SRC}/train_grpo.py"
                f" --model_name {MODEL_HF}"
                f" --output_dir {outdir}"
                f" --dataset_name {GSM8K_DATASET}"
                f" --dataset_config {GSM8K_CONFIG}"
                f" --reward_type {reward_type}"
                f" --num_generations {gsize}"
                f" --kl_coef {kl}"
                f" --temperature {GRPO_TEMPERATURE}"
                f" --learning_rate {GRPO_LR}"
                f" --lora_r {GRPO_LORA_R}"
                f" --lora_alpha {GRPO_LORA_ALPHA}"
                f" --num_train_epochs 1"
                f" --per_device_batch_size {GRPO_BATCH}"
                f" --gradient_accumulation_steps {GRPO_GRAD_ACCUM}"
                f" --max_prompt_length {GRPO_MAX_PROMPT}"
                f" --max_completion_length {GRPO_MAX_COMPLETION}"
                f" --max_train_samples 500"
                f" --logging_steps 5 --save_steps 100"
            )
            if use_sft:
                cmd += f" --sft_adapter_path {SFT_ADAPTER}"

            t0 = time.time()
            !{cmd}
            dt = (time.time()-t0)/60
            adapter = f"{outdir}/final_grpo"
            if os.path.exists(adapter):
                ablation_results[name] = {"status":"ok","adapter":adapter,"time_min":round(dt,1)}
                print(f"✅ {name} ({dt:.1f} min)")
            else:
                ablation_results[name] = {"status":"no_adapter"}
        except Exception as e:
            ablation_results[name] = {"status":"failed","error":str(e)[:150]}
            print(f"❌ {name}: {str(e)[:150]}")

    with open(f"{OUTPUT_ROOT}/ablation_summary.json","w") as f:
        json.dump(ablation_results, f, indent=2, ensure_ascii=False)
    ok = sum(1 for v in ablation_results.values() if v["status"]=="ok")
    print(f"\n消融: {ok}/{len(ablations)} 成功")

from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from reward import extract_final_answer, check_thinking_format, count_reasoning_steps, is_correct

MODEL_MAP = {"qwen3.5-0.8b": "Qwen/Qwen3.5-0.8B", "qwen3.5-1.7b": "Qwen/Qwen3.5-1.7B"}

# ---- 评估任务 ----
eval_tasks = [
    ("base_no_think", MODEL_HF, None, False),
    ("base_think",    MODEL_HF, None, True),
]
if SFT_ADAPTER and os.path.exists(SFT_ADAPTER):
    eval_tasks.append(("sft", MODEL_HF, SFT_ADAPTER, True))
if GRPO_GSM8K_ADAPTER and os.path.exists(GRPO_GSM8K_ADAPTER):
    eval_tasks.append(("grpo_gsm8k", MODEL_HF, GRPO_GSM8K_ADAPTER, True))
if GRPO_MIXED_ADAPTER and os.path.exists(GRPO_MIXED_ADAPTER):
    eval_tasks.append(("grpo_mixed", MODEL_HF, GRPO_MIXED_ADAPTER, True))
for name, info in ablation_results.items():
    if info.get("status") == "ok" and info.get("adapter") and os.path.exists(info["adapter"]):
        eval_tasks.append((name, MODEL_HF, info["adapter"], True))

print(f"评估: {len(eval_tasks)} 个模型")

In [ ]:
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from reward import extract_final_answer, check_thinking_format, count_reasoning_steps, is_correct

MODEL_MAP = {"qwen3.5-0.8b": "Qwen/Qwen3.5-0.8B-Instruct", "qwen3.5-2b": "Qwen/Qwen3.5-2B-Instruct"}

# ---- 评估任务 ----
eval_tasks = [
    ("base_no_think", MODEL_HF, None, False),
    ("base_think",    MODEL_HF, None, True),
]
if SFT_ADAPTER and os.path.exists(SFT_ADAPTER):
    eval_tasks.append(("sft", MODEL_HF, SFT_ADAPTER, True))
if GRPO_GSM8K_ADAPTER and os.path.exists(GRPO_GSM8K_ADAPTER):
    eval_tasks.append(("grpo_gsm8k", MODEL_HF, GRPO_GSM8K_ADAPTER, True))
if GRPO_MIXED_ADAPTER and os.path.exists(GRPO_MIXED_ADAPTER):
    eval_tasks.append(("grpo_mixed", MODEL_HF, GRPO_MIXED_ADAPTER, True))
for name, info in ablation_results.items():
    if info.get("status") == "ok" and info.get("adapter") and os.path.exists(info["adapter"]):
        eval_tasks.append((name, MODEL_HF, info["adapter"], True))

print(f"评估: {len(eval_tasks)} 个模型")

# ---- 评估函数 ----
def run_eval(model, tokenizer, dataset, enable_thinking):
    n = min(EVAL_MAX_SAMPLES, len(dataset))
    correct = has_thinking = total_steps = 0
    results = []
    for i in tqdm(range(n), desc="eval"):
        ex = dataset[i]
        q = ex.get("question") or ex.get("problem_text", "")
        gt_raw = ex.get("answer") or ex.get("solution") or ""
        if "####" in str(gt_raw):
            gt = str(gt_raw).split("####")[-1].strip()
        else:
            ans = ex.get("answer_number", ex.get("answer_latex", ""))
            unit = ex.get("unit", "")
            gt = f"{ans} {unit}".strip() if unit else str(ans)
        if not q or not gt:
            continue

        sys = "一步步思考，将推理写在<think>和</think>之间，最后给答案。" if enable_thinking else "直接给答案。"
        prompt = f"<|im_start|>system\n{sys}<|im_end|>\n<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=EVAL_MAX_NEW_TOKENS, temperature=0.7, do_sample=True, top_p=0.95, pad_token_id=tokenizer.pad_token_id)
        resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred = extract_final_answer(resp)
        c = is_correct(pred, gt)
        t = check_thinking_format(resp)
        if c: correct += 1
        if t: has_thinking += 1; total_steps += count_reasoning_steps(resp)
        results.append({"q":q[:100],"gt":gt,"pred":pred,"correct":c,"thinking":t})

    total = len(results)
    return {"accuracy":round(correct/total*100,2) if total else 0,
            "thinking_rate":round(has_thinking/total*100,2) if total else 0,
            "avg_reasoning_steps":round(total_steps/has_thinking,2) if has_thinking else 0,
            "total":total,"correct":correct}

# ---- 运行 ----
eval_metrics_all = {}
gsm8k_test = load_dataset(GSM8K_DATASET, GSM8K_CONFIG, split="test") if EVAL_GSM8K else None
scibench_test = Dataset.from_list(json.load(open(SCIBENCH_TEST_FILE))) if EVAL_SCIBENCH and os.path.exists(SCIBENCH_TEST_FILE) else None

for label, model_name, adapter, thinking in eval_tasks:
    print(f"\n{'='*60}\n评估: {label} | thinking={thinking}\n{'='*60}")

    hf_name = MODEL_MAP.get(model_name, model_name)
    tokenizer = AutoTokenizer.from_pretrained(hf_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(hf_name, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    if adapter and os.path.exists(adapter):
        model = PeftModel.from_pretrained(model, adapter)
    model.eval()

    entry = {}
    if gsm8k_test:
        m = run_eval(model, tokenizer, gsm8k_test, thinking)
        entry["gsm8k"] = m
        print(f"  GSM8K: acc={m['accuracy']}% think={m['thinking_rate']}% steps={m['avg_reasoning_steps']}")
    if scibench_test:
        print(f"  SciBench held-out: {len(scibench_test)} 题")
        m = run_eval(model, tokenizer, scibench_test, thinking)
        entry["scibench"] = m
        print(f"  SciBench: acc={m['accuracy']}% think={m['thinking_rate']}% steps={m['avg_reasoning_steps']}")

    if entry:
        eval_metrics_all[label] = entry
    del model; torch.cuda.empty_cache()

with open(f"{OUTPUT_ROOT}/all_eval_metrics.json","w") as f:
    json.dump(eval_metrics_all, f, indent=2, ensure_ascii=False)
print(f"\n✅ 评估完成 → {OUTPUT_ROOT}/all_eval_metrics.json")

---
## Phase 6: 结果汇总

In [ ]:
if not eval_metrics_all:
    print("⚠️  无评估数据")
else:
    ds_names = list(list(eval_metrics_all.values())[0].keys())
    fig, axes = plt.subplots(1, len(ds_names), figsize=(8*len(ds_names), 6))
    if len(ds_names) == 1:
        axes = [axes]

    for idx, ds_name in enumerate(ds_names):
        rows = []
        for label, metrics in eval_metrics_all.items():
            m = metrics.get(ds_name, {})
            if m:
                rows.append({"方法": label, "准确率 (%)": m.get("accuracy", 0)})
        if not rows:
            axes[idx].set_title(f"{ds_name} (无数据)")
            continue
        df = pd.DataFrame(rows).sort_values("准确率 (%)", ascending=True)
        colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(df)))
        axes[idx].barh(df["方法"], df["准确率 (%)"], color=colors)
        for i, v in enumerate(df["准确率 (%)"].values):
            axes[idx].text(v+0.3, i, f"{v:.1f}", va="center", fontsize=9)
        axes[idx].set_xlabel("Accuracy (%)")
        axes[idx].set_title(f"{ds_name.upper()}")

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_ROOT}/results_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

    # 文本汇总
    for ds_name in ds_names:
        print(f"\n--- {ds_name.upper()} ---")
        print(f"{'方法':<22} {'准确率':>8} {'思考率':>8} {'步数':>6}")
        ds_rows = []
        for label, metrics in eval_metrics_all.items():
            m = metrics.get(ds_name, {})
            if m:
                ds_rows.append((label, m.get("accuracy",0), m.get("thinking_rate",0), m.get("avg_reasoning_steps",0)))
        for row in sorted(ds_rows, key=lambda x: -x[1]):
            print(f"{row[0]:<22} {row[1]:>7.1f}% {row[2]:>7.1f}% {row[3]:>5.1f}")

    print(f"\n📈 {OUTPUT_ROOT}/results_comparison.png")

---
## 🎉 全部完成

| # | Baseline | 训练数据 |
|---|----------|--------|
| ① | base_no_think | 无 |
| ② | base_think | 无 |
| ③ | sft | GSM8K |
| ④ | grpo_gsm8k | GSM8K |
| ⑤ | **grpo_mixed** | **GSM8K + SciBench 7科** |

测试集: GSM8K test + SciBench held-out 3科 (quan/stat/thermo)